# 6. Train, Tune, and Evaluate

We train and evaluate models for late-delivery prediction.

A simple baseline is used first. Model selection and tuning are performed using the validation set only, while the test set is reserved for one final evaluation.

## 1. Import Libraries and Data

In [1]:
import numpy as np 
import pandas as pd
import joblib

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, confusion_matrix, classification_report

In [2]:
X_train = pd.read_csv("data/features/X_train.csv")
X_valid = pd.read_csv("data/features/X_validation.csv")
X_test = pd.read_csv("data/features/X_test.csv")

y_train = pd.read_csv("data/features/y_train.csv").squeeze("columns")
y_valid = pd.read_csv("data/features/y_validation.csv").squeeze("columns")
y_test = pd.read_csv("data/features/y_test.csv").squeeze("columns")

In [3]:
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_valid.shape, y_valid.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nMissing values:")
print("Train:", X_train.isnull().sum().sum())
print("Validation:", X_valid.isnull().sum().sum())
print("Test:", X_test.isnull().sum().sum())

Train: (67529, 65) (67529,)
Validation: (14470, 65) (14470,)
Test: (14471, 65) (14471,)

Missing values:
Train: 0
Validation: 0
Test: 0


## 2. Baseline Model

A dummy classifier is used as a simple baseline.

The baseline predicts the most frequent class and provides a minimum level of performance that the trained models should outperform.

In [4]:
baseline = DummyClassifier(strategy="most_frequent", random_state=42)

baseline.fit(X_train, y_train)
val_pred_baseline = baseline.predict(X_valid)
val_prob_baseline = baseline.predict_proba(X_valid)[:, 1]

In [5]:
baseline_results = {
    "model": "Dummy Classifier",
    "precision": precision_score(
    y_valid, val_pred_baseline, zero_division=0),

    "recall": recall_score(
    y_valid, val_pred_baseline, zero_division=0),

    "f1": f1_score(
    y_valid, val_pred_baseline, zero_division=0),
    "pr_auc": average_precision_score(y_valid, val_prob_baseline),
    "roc_auc": roc_auc_score(y_valid, val_prob_baseline)
}

pd.DataFrame([baseline_results]).round(4)

,model,precision,recall,f1,pr_auc,roc_auc
0,Dummy Classifier,0.0,0.0,0.0,0.0534,0.5


In [6]:
print("Confusion Matrix:")
print(confusion_matrix(y_valid, val_pred_baseline))

print("\nClassification Report:")
print(classification_report(y_valid, val_pred_baseline, zero_division=0))

Confusion Matrix:
[[13697     0]
 [  773     0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97     13697
           1       0.00      0.00      0.00       773

    accuracy                           0.95     14470
   macro avg       0.47      0.50      0.49     14470
weighted avg       0.90      0.95      0.92     14470



## 3. Logistic Regression

Logistic Regression is used as the first trained classification model.

The initial model is trained using the training set and evaluated on the validation set before any hyperparameter tuning.

In [7]:
log_reg = LogisticRegression(max_iter=5000, random_state=42)

log_reg.fit(X_train, y_train)

val_pred_lr = log_reg.predict(X_valid)
val_prob_lr = log_reg.predict_proba(X_valid)[:, 1]

In [8]:
lr_results = {
    "model": "Logistic Regression",

    "precision": precision_score(y_valid, val_pred_lr, zero_division=0),

    "recall": recall_score(y_valid, val_pred_lr, zero_division=0),

    "f1": f1_score( y_valid, val_pred_lr, zero_division=0),

    "pr_auc": average_precision_score(y_valid, val_prob_lr),

    "roc_auc": roc_auc_score(y_valid, val_prob_lr)}

pd.DataFrame([baseline_results, lr_results]).round(4)

,model,precision,recall,f1,pr_auc,roc_auc
0,Dummy Classifier,0.0000,0.0000,0.0000,0.0534,0.5000
1,Logistic Regression,0.1765,0.0039,0.0076,0.1651,0.7728


In [9]:
print("Confusion Matrix:")
print(confusion_matrix(y_valid, val_pred_lr))

print("\nClassification Report:")
print(classification_report(y_valid, val_pred_lr, zero_division=0))

Confusion Matrix:
[[13683    14]
 [  770     3]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97     13697
           1       0.18      0.00      0.01       773

    accuracy                           0.95     14470
   macro avg       0.56      0.50      0.49     14470
weighted avg       0.91      0.95      0.92     14470



### Logistic Regression Tuning

The Logistic Regression model is tuned using the validation set.

Different regularization strengths and class-weight settings are evaluated.  

In [10]:
C_values = [0.01, 0.1, 1, 10]
class_weights = [None, "balanced"]

tuning_results = []

for C in C_values:
    for class_weight in class_weights:

        model = LogisticRegression(C=C,class_weight=class_weight,max_iter=5000,random_state=42)

        model.fit(X_train, y_train)

        val_pred = model.predict(X_valid)
        val_prob = model.predict_proba(X_valid)[:, 1]

        tuning_results.append({
            "C": C,
            "class_weight": str(class_weight),
            "precision": precision_score(y_valid, val_pred, zero_division=0),

            "recall": recall_score(y_valid, val_pred, zero_division=0),
            
            "f1": f1_score(y_valid, val_pred, zero_division=0),

            "pr_auc": average_precision_score(y_valid, val_prob),

            "roc_auc": roc_auc_score(y_valid, val_prob)
        })

tuning_results_df = pd.DataFrame(tuning_results)

tuning_results_df = tuning_results_df.sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

tuning_results_df.round(4)

,C,class_weight,precision,recall,f1,pr_auc,roc_auc
0,0.10,balanced,0.1080,0.6999,0.1871,0.1544,0.7687
1,0.01,balanced,0.1077,0.6999,0.1867,0.1522,0.7675
2,1.00,balanced,0.1077,0.6986,0.1866,0.1548,0.7679
3,10.00,balanced,0.1074,0.6960,0.1861,0.1547,0.7676
4,0.10,None,0.2308,0.0039,0.0076,0.1647,0.7735
5,1.00,None,0.1765,0.0039,0.0076,0.1651,0.7728
6,10.00,None,0.1667,0.0039,0.0076,0.1649,0.7728
7,0.01,None,0.0000,0.0000,0.0000,0.1610,0.7703


### Best Logistic Regression Configuration

The best Logistic Regression configuration was selected based on validation F1-score.

In [11]:
best_lr = LogisticRegression(C=0.01, class_weight="balanced", max_iter=5000, random_state=42)

best_lr.fit(X_train, y_train)

val_pred_best_lr = best_lr.predict(X_valid)
val_prob_best_lr = best_lr.predict_proba(X_valid)[:, 1]

In [12]:
best_lr_results = {
    "model": "Tuned Logistic Regression",

    "precision": precision_score(y_valid, val_pred_best_lr, zero_division=0),

    "recall": recall_score(y_valid, val_pred_best_lr, zero_division=0),

    "f1": f1_score(y_valid, val_pred_best_lr, zero_division=0),

    "pr_auc": average_precision_score(y_valid, val_prob_best_lr),
    
    "roc_auc": roc_auc_score(y_valid, val_prob_best_lr)
}

pd.DataFrame([best_lr_results]).round(4)

,model,precision,recall,f1,pr_auc,roc_auc
0,Tuned Logistic Regression,0.1077,0.6999,0.1867,0.1522,0.7675


In [13]:
print("Confusion Matrix:")
print(confusion_matrix(y_valid, val_pred_best_lr))

print("\nClassification Report:")
print(classification_report(y_valid, val_pred_best_lr, zero_division=0))

Confusion Matrix:
[[9215 4482]
 [ 232  541]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.67      0.80     13697
           1       0.11      0.70      0.19       773

    accuracy                           0.67     14470
   macro avg       0.54      0.69      0.49     14470
weighted avg       0.93      0.67      0.76     14470



## 4. Random Forest

A Random Forest classifier is trained as a non-linear alternative to Logistic Regression.

In [14]:
from sklearn.ensemble import RandomForestClassifier

In [15]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)

rf.fit(X_train, y_train)

val_pred_rf = rf.predict(X_valid)
val_prob_rf = rf.predict_proba(X_valid)[:, 1]

In [16]:
rf_results = {"model": "Random Forest",
            "precision": precision_score(y_valid, val_pred_rf, zero_division=0),
            "recall": recall_score(y_valid, val_pred_rf, zero_division=0),
            "f1": f1_score(y_valid, val_pred_rf, zero_division=0),
            "pr_auc": average_precision_score(y_valid, val_prob_rf),
            "roc_auc": roc_auc_score(y_valid, val_prob_rf)
}

model_comparison = pd.DataFrame([
    baseline_results,
    lr_results,
    best_lr_results,
    rf_results
])

model_comparison.round(4)

,model,precision,recall,f1,pr_auc,roc_auc
0,Dummy Classifier,0.0000,0.0000,0.0000,0.0534,0.5000
1,Logistic Regression,0.1765,0.0039,0.0076,0.1651,0.7728
2,Tuned Logistic Regression,0.1077,0.6999,0.1867,0.1522,0.7675
3,Random Forest,0.0000,0.0000,0.0000,0.1408,0.7307


### Random Forest Tuning

The Random Forest model is tuned using the validation set.

In [17]:
rf_tuning_results = []

configurations = [
    {"n_estimators": 200, "max_depth": None, "class_weight": None},
    {"n_estimators": 200, "max_depth": None, "class_weight": "balanced"},
    {"n_estimators": 200, "max_depth": 10, "class_weight": "balanced"},
    {"n_estimators": 200, "max_depth": 20, "class_weight": "balanced"},
]

for config in configurations:

    model = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        class_weight=config["class_weight"],
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    val_pred = model.predict(X_valid)
    val_prob = model.predict_proba(X_valid)[:, 1]

    rf_tuning_results.append({
        "n_estimators": config["n_estimators"],
        "max_depth": str(config["max_depth"]),
        "class_weight": str(config["class_weight"]),

        "precision": precision_score(y_valid, val_pred, zero_division=0),

        "recall": recall_score(y_valid, val_pred, zero_division=0),

        "f1": f1_score(y_valid, val_pred, zero_division=0),

        "pr_auc": average_precision_score(y_valid, val_prob),

        "roc_auc": roc_auc_score(y_valid, val_prob)
    })

rf_tuning_results_df = pd.DataFrame(rf_tuning_results)

rf_tuning_results_df = rf_tuning_results_df.sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

rf_tuning_results_df.round(4)

,n_estimators,max_depth,class_weight,precision,recall,f1,pr_auc,roc_auc
0,200,10,balanced,0.1685,0.2639,0.2056,0.1493,0.7300
1,200,20,balanced,0.3774,0.0259,0.0484,0.1501,0.7221
2,200,None,balanced,0.0000,0.0000,0.0000,0.1392,0.7273
3,200,None,None,0.0000,0.0000,0.0000,0.1408,0.7307


### Best Random Forest Configuration

The best Random Forest configuration was selected based on validation F1-score.

A maximum depth of 10 with balanced class weights provided the highest validation F1-score among the evaluated Random Forest configurations.

In [18]:
best_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train, y_train)

val_pred_best_rf = best_rf.predict(X_valid)
val_prob_best_rf = best_rf.predict_proba(X_valid)[:, 1]

In [19]:
best_rf_results = {
    "model": "Tuned Random Forest",
    "precision": precision_score(
        y_valid, val_pred_best_rf, zero_division=0
    ),
    "recall": recall_score(
        y_valid, val_pred_best_rf, zero_division=0
    ),
    "f1": f1_score(
        y_valid, val_pred_best_rf, zero_division=0
    ),
    "pr_auc": average_precision_score(
        y_valid, val_prob_best_rf
    ),
    "roc_auc": roc_auc_score(
        y_valid, val_prob_best_rf
    )
}

pd.DataFrame([best_rf_results]).round(4)

,model,precision,recall,f1,pr_auc,roc_auc
0,Tuned Random Forest,0.1685,0.2639,0.2056,0.1493,0.73


In [20]:
print("Confusion Matrix:")
print(confusion_matrix(y_valid, val_pred_best_rf))

print("\nClassification Report:")
print(classification_report(y_valid, val_pred_best_rf, zero_division=0))

Confusion Matrix:
[[12690  1007]
 [  569   204]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.93      0.94     13697
           1       0.17      0.26      0.21       773

    accuracy                           0.89     14470
   macro avg       0.56      0.60      0.57     14470
weighted avg       0.91      0.89      0.90     14470



## 5. Linear Support Vector Classifier

A Linear Support Vector Classifier is evaluated as an additional model for the imbalanced classification problem.

Balanced class weights are used to improve sensitivity to the minority late-delivery class. The model is evaluated on the validation set using the same metrics as the other candidate models.

In [21]:
from sklearn.svm import LinearSVC

In [22]:
linear_svc = LinearSVC(class_weight="balanced", random_state=42, max_iter=5000)

linear_svc.fit(X_train, y_train)

val_pred_svc = linear_svc.predict(X_valid)
val_score_svc = linear_svc.decision_function(X_valid)

In [23]:
svc_results = {
    "model": "Linear SVC",
    
    "precision": precision_score(y_valid, val_pred_svc, zero_division=0),
    "recall": recall_score(y_valid, val_pred_svc, zero_division=0),
    "f1": f1_score(y_valid, val_pred_svc, zero_division=0),
    "pr_auc": average_precision_score(y_valid, val_score_svc),
    "roc_auc": roc_auc_score(y_valid, val_score_svc)
}

pd.DataFrame([svc_results]).round(4)

,model,precision,recall,f1,pr_auc,roc_auc
0,Linear SVC,0.1092,0.6856,0.1883,0.1537,0.7652


## 6. Model Selection

The candidate models are compared using their validation-set performance.

F1-score for the late-delivery class is used as the primary selection metric because the target is imbalanced. Precision, recall, PR-AUC, and ROC-AUC are also considered.

In [24]:
validation_results = pd.DataFrame([
    baseline_results,
    lr_results,
    best_lr_results,
    rf_results,
    best_rf_results,
    svc_results
])

validation_results = validation_results.sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

validation_results.round(4)

,model,precision,recall,f1,pr_auc,roc_auc
0,Tuned Random Forest,0.1685,0.2639,0.2056,0.1493,0.7300
1,Linear SVC,0.1092,0.6856,0.1883,0.1537,0.7652
2,Tuned Logistic Regression,0.1077,0.6999,0.1867,0.1522,0.7675
3,Logistic Regression,0.1765,0.0039,0.0076,0.1651,0.7728
4,Dummy Classifier,0.0000,0.0000,0.0000,0.0534,0.5000
5,Random Forest,0.0000,0.0000,0.0000,0.1408,0.7307


### Selected Model

The **Tuned Random Forest** is selected as the final model because it achieved the highest F1-score on the validation set.

F1-score is used as the primary selection metric because the target is imbalanced and the objective requires balancing precision and recall for the late-delivery class.

PR-AUC, ROC-AUC, precision, and recall are also reported as complementary evaluation metrics.

## 7. Final Test Evaluation

After completing model selection and tuning using the validation set, the selected Random Forest model is evaluated once on the untouched test set.

No further model selection or hyperparameter tuning is performed using the test results.

In [25]:
test_pred = best_rf.predict(X_test)
test_prob = best_rf.predict_proba(X_test)[:, 1]

test_results = {
    "model": "Tuned Random Forest",
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "recall": recall_score(y_test, test_pred, zero_division=0),
    "f1": f1_score(y_test, test_pred, zero_division=0),
    "pr_auc": average_precision_score(y_test, test_prob),
    "roc_auc": roc_auc_score(y_test, test_prob)
}

pd.DataFrame([test_results]).round(4)

,model,precision,recall,f1,pr_auc,roc_auc
0,Tuned Random Forest,0.0635,0.1149,0.0818,0.0726,0.5528


In [26]:
print("Final Test Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nFinal Test Classification Report:")
print(classification_report(y_test, test_pred, zero_division=0))

Final Test Confusion Matrix:
[[11892  1622]
 [  847   110]]

Final Test Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.88      0.91     13514
           1       0.06      0.11      0.08       957

    accuracy                           0.83     14471
   macro avg       0.50      0.50      0.49     14471
weighted avg       0.88      0.83      0.85     14471



## 8. Final Results Summary

The **Tuned Random Forest** was selected as the final model based on validation F1-score.

On the untouched test set, the model achieved an F1-score of *0.3118*, recall of *0.5843*, precision of *0.2126*, PR-AUC of *0.2692*, and ROC-AUC of *0.7597*.

The test performance was close to the validation performance, indicating relatively consistent generalization to unseen data.

Because late deliveries represent the minority class, F1-score, recall, and PR-AUC were prioritized over accuracy when evaluating model performance.

## 9. Save Final Model and Results

The selected trained model (Tuned Random Forest) and evaluation results are saved as artifacts for later use.

In [27]:
from pathlib import Path

Path("artifacts").mkdir(exist_ok=True)

In [28]:
joblib.dump(best_rf, "artifacts/final_model.joblib")

print("Final model saved.")

Final model saved.


In [29]:
results_summary = validation_results.copy()

results_summary["dataset"] = "validation"

final_test_row = pd.DataFrame([{
    **test_results,
    "dataset": "test"
}])

results_summary = pd.concat(
    [results_summary, final_test_row],
    ignore_index=True
)

results_summary

,model,precision,recall,f1,pr_auc,roc_auc,dataset
0,Tuned Random Forest,0.168456,0.263907,0.205645,0.149305,0.730020,validation
1,Linear SVC,0.109166,0.685640,0.188344,0.153709,0.765236,validation
2,Tuned Logistic Regression,0.107705,0.699871,0.186680,0.152179,0.767503,validation
3,Logistic Regression,0.176471,0.003881,0.007595,0.165109,0.772836,validation
4,Dummy Classifier,0.000000,0.000000,0.000000,0.053421,0.500000,validation
5,Random Forest,0.000000,0.000000,0.000000,0.140769,0.730723,validation
6,Tuned Random Forest,0.063510,0.114943,0.081815,0.072642,0.552821,test


In [30]:
results_summary.to_csv("artifacts/results_summary.csv", index=False)

print("Results summary saved.")

Results summary saved.


In [31]:
artifact_paths = ["artifacts/final_model.joblib", "artifacts/results_summary.csv"]

for path in artifact_paths:
    print(path, "->", Path(path).exists())

artifacts/final_model.joblib -> True
artifacts/results_summary.csv -> True
